In [1]:
from transformers import AutoTokenizer
from pathlib import Path
import pandas as pd

In [2]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B", trust_remote_code=True)

In [3]:
def count_tokens_in_directory(directory, extension=".md"):
    total_tokens = 0
    file_stats = []
    
    for file_path in Path(directory).rglob(f"*{extension}"):
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
        
        tokens = tokenizer.encode(content)
        token_count = len(tokens)
        total_tokens += token_count
        
        file_stats.append({
            "file": str(file_path),
            "tokens": token_count,
            "chars": len(content)
        })
    
    return total_tokens, file_stats

In [4]:
total, stats = count_tokens_in_directory("../../data/parsed")
stats_df = pd.DataFrame(stats)

In [5]:
stats_df.describe()

,tokens,chars
count,56.000000,56.000000
mean,10766.750000,35929.535714
std,3651.442585,10841.271437
min,4891.000000,17872.000000
25%,8119.750000,28345.250000
50%,10459.500000,34809.000000
75%,12979.000000,43409.500000
max,20405.000000,66180.000000


In [6]:
after_df = stats_df.copy()

total_before, stats_before = count_tokens_in_directory("../local_data/parsed/gemini_3_flash")
before_df = pd.DataFrame(stats_before)

result_df = before_df.join(after_df, lsuffix="_before", rsuffix="_after")
result_df['diff'] = result_df['tokens_after'] - result_df['tokens_before']

result_df[['tokens_before', 'tokens_after', 'diff']].describe()

,tokens_before,tokens_after,diff
count,56.000000,56.000000,56.000000
mean,13730.232143,10766.750000,-2963.482143
std,4482.862726,3651.442585,1904.694363
min,6049.000000,4891.000000,-8895.000000
25%,10377.250000,8119.750000,-4396.000000
50%,13215.500000,10459.500000,-2849.500000
75%,16182.500000,12979.000000,-1770.500000
max,27635.000000,20405.000000,0.000000
